# Task 2: QLoRA Fine-Tuning Qwen2.5-Coder-7B-Instruct
- Dataset: 477 synthetic balanced samples (Silver V2) + 32 Gold samples (upweighted x2) = 541 ChatML records.
- Method: QLoRA 4-bit SFT with completion-only loss masking.
- Output: `/kaggle/working/task2_predictions.json` ready for submission.

In [ ]:
!nvidia-smi
!pip install -q bitsandbytes>=0.42.0 peft>=0.9.0 accelerate>=0.27.0 transformers>=4.38.0

In [ ]:
import os, sys, shutil
from pathlib import Path

# Ensure project repo is loaded
repo_dir = Path("/kaggle/working/repo")
if not repo_dir.exists():
    !git clone -b task-2-split-run https://github.com/tiendinhquang2104/llm-final-project.git /kaggle/working/repo
else:
    !cd /kaggle/working/repo && git pull origin task-2-split-run

os.chdir("/kaggle/working/repo")
sys.path.insert(0, "/kaggle/working/repo")
print("Working directory:", os.getcwd())

In [ ]:
# Step 1: Prepare training data (combining synth + gold)
!python scripts/prepare_task2_sft.py

In [ ]:
# Step 2: Train LoRA adapter on Qwen2.5-Coder-7B-Instruct (approx. 15-20 mins on T4)
!python scripts/train_task2_sft.py \
    --model_id Qwen/Qwen2.5-Coder-7B-Instruct \
    --data_path data/processed/task2_sft_train.jsonl \
    --output_dir /kaggle/working/qlora_adapter \
    --epochs 3 \
    --batch_size 2 \
    --grad_accum 4 \
    --lr 2e-4

In [ ]:
# Step 3: Run inference on Public Test
!python scripts/infer_task2_sft.py \
    --model_id Qwen/Qwen2.5-Coder-7B-Instruct \
    --adapter_dir /kaggle/working/qlora_adapter \
    --data_dir data/public_test_release \
    --output_file /kaggle/working/task2_predictions.json

print("Done! Output saved to /kaggle/working/task2_predictions.json")

In [ ]:
# Step 4: Verify submission format and inspect label distribution
import json
from collections import Counter

with open('/kaggle/working/task2_predictions.json', encoding='utf-8') as f:
    preds = json.load(f)

print(f'Total predicted samples: {len(preds)}')
assert len(preds) == 328, f'Expected 328 samples, found {len(preds)}'

all_labels = [l for p in preds for l in p['output']['taxonomy_error']]
counts = Counter(all_labels)
print(f'\nPredicted taxonomy distribution (total labels: {len(all_labels)}):')
for label, count in sorted(counts.items(), key=lambda x: -x[1]):
    print(f'  - {label}: {count}')
